# Audit delle singole run: anomalie e casi interessanti per la tesi

Questo notebook legge direttamente `run_config.json`, `metrics.json`, `test_metrics.json`
e `history.jsonl` dalle 640 directory di run contenute in:

- `colab_state.tar.gz` per il training set completo;
- `colab_subset_state.tar.gz` per 100, 300 e 900 immagini.

Lo scopo non è eliminare automaticamente osservazioni, ma individuare run da esaminare:
outlier prestazionali, discrepanze validation–test, dinamiche di training insolite, gradienti
quantistici sospetti, rallentamenti e seed che modificano fortemente i confronti appaiati.

Le anomalie vengono cercate **solo dentro gruppi omogenei** per task, regime, modello e qubit.
Le segnalazioni sono esplorative e non introducono nuovi p-value.

In [ ]:
# Configurazione: modificare soltanto questi percorsi se gli archivi sono altrove.
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

DEFAULT_BUNDLE_DIR = (
    Path("/content/drive/MyDrive/tesi_qml_colab/colab_bundle")
    if IN_COLAB else Path.cwd()
)
FULL_ARCHIVE = Path(os.environ.get(
    "TESI_FULL_ARCHIVE", DEFAULT_BUNDLE_DIR / "colab_state.tar.gz"
))
SUBSET_ARCHIVE = Path(os.environ.get(
    "TESI_SUBSET_ARCHIVE", DEFAULT_BUNDLE_DIR / "colab_subset_state.tar.gz"
))
OUTPUT_DIR = Path(os.environ.get(
    "TESI_ANOMALY_OUTPUT",
    "/content/drive/MyDrive/tesi_qml_colab/run_anomaly_audit"
    if IN_COLAB else Path.cwd() / "run_anomaly_audit",
))
FAST_MODE = os.environ.get("TESI_ANALYSIS_FAST", "0") == "1"
MAKE_ZIP = os.environ.get("TESI_ANALYSIS_ZIP", "1") != "0"
EXPORT_DPI = 90 if FAST_MODE else 300
TOP_CASES = 6

for archive_path in (FULL_ARCHIVE, SUBSET_ARCHIVE):
    if not archive_path.is_file():
        raise FileNotFoundError(
            f"Archivio non trovato: {archive_path}\n"
            "Modifica FULL_ARCHIVE e SUBSET_ARCHIVE nella prima cella."
        )

FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print({
    "full_archive": str(FULL_ARCHIVE),
    "subset_archive": str(SUBSET_ARCHIVE),
    "output": str(OUTPUT_DIR),
    "dpi": EXPORT_DPI,
})

In [ ]:
# Stack esclusivamente dedicato ad analisi dati e visualizzazione.
import json
import math
import shutil
import tarfile
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import PurePosixPath

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(context="paper", style="white", font="DejaVu Sans")
matplotlib.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": EXPORT_DPI,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "axes.titleweight": "semibold",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
    "axes.axisbelow": True,
})

REGIME_ORDER = ("100", "300", "900", "full")
REGIME_LABELS = {"100": "100", "300": "300", "900": "900", "full": "Full"}
TASK_LABELS = {"binary": "Binario", "three_class": "Tre classi"}
MODEL_COLORS = {
    "B0": "#9A9A9A", "B1": "#5F6368", "B2": "#0072B2",
    "QF": "#E69F00", "Q": "#CC79A7",
}
SEVERITY_SCORE = {"low": 1, "medium": 2, "high": 3, "critical": 4}
TARGET_FILES = {"run_config.json", "metrics.json", "test_metrics.json", "history.jsonl"}
EXPORTED_FIGURES = []
EXPORTED_TABLES = []

## 1. Lettura sequenziale dei JSON dagli archivi

La scansione attraversa ciascun TAR una sola volta. Soltanto i quattro file diagnostici di
ogni run vengono decodificati; checkpoint, Parquet delle predizioni e feature vengono saltati.
L'avanzamento viene stampato ogni 250 file utili, così la cella non appare bloccata.

In [ ]:
def normalize_subset(value):
    if value is None:
        return "full"
    text = str(value).strip().lower()
    return text[:-2] if text.endswith(".0") else text


def scan_run_jsons(archive_path, allowed_subsets, archive_label):
    bundles = defaultdict(dict)
    parse_issues = []
    useful_files = 0
    print(f"Scansione {archive_label}: {archive_path.name}", flush=True)
    with tarfile.open(archive_path, "r:gz") as archive:
        for member in archive:
            if not member.isfile():
                continue
            normalized = PurePosixPath(member.name).as_posix().lstrip("./")
            path = PurePosixPath(normalized)
            if "results" not in path.parts or "runs" not in path.parts:
                continue
            if path.name not in TARGET_FILES:
                continue
            extracted = archive.extractfile(member)
            if extracted is None:
                parse_issues.append({
                    "archive": archive_label, "run_path": str(path.parent),
                    "file": path.name, "issue": "membro TAR illeggibile",
                })
                continue
            payload = extracted.read().decode("utf-8")
            try:
                if path.name == "history.jsonl":
                    parsed = [
                        json.loads(line) for line in payload.splitlines() if line.strip()
                    ]
                else:
                    parsed = json.loads(payload)
            except Exception as error:
                parsed = None
                parse_issues.append({
                    "archive": archive_label, "run_path": str(path.parent),
                    "file": path.name, "issue": f"JSON non valido: {error}",
                })
            bundles[str(path.parent)][path.name] = parsed
            useful_files += 1
            if useful_files % 250 == 0:
                print(f"  {useful_files} file JSON/JSONL letti...", flush=True)

    selected = []
    for run_path, bundle in bundles.items():
        config = bundle.get("run_config.json")
        subset = normalize_subset(config.get("subset_size")) if isinstance(config, dict) else None
        if subset in allowed_subsets:
            selected.append({
                "archive": archive_label,
                "run_path": run_path,
                "bundle": bundle,
            })
    print(
        f"Completato {archive_label}: {useful_files} file utili, "
        f"{len(selected)} run selezionate, {len(parse_issues)} problemi di parsing.",
        flush=True,
    )
    return selected, parse_issues


full_bundles, full_parse_issues = scan_run_jsons(
    FULL_ARCHIVE, {"full"}, "full"
)
subset_bundles, subset_parse_issues = scan_run_jsons(
    SUBSET_ARCHIVE, {"100", "300", "900"}, "subset"
)
run_bundles = full_bundles + subset_bundles
parse_issues = full_parse_issues + subset_parse_issues
print({"run_bundles": len(run_bundles), "parse_issues": len(parse_issues)})

## 2. Profilo di ogni run e della dinamica di training

Ogni riga della tabella risultante corrisponde a una run. Le metriche `metrics.json` sono
trattate come validation, mentre `test_metrics.json` rappresenta il test finale. Dalla storia
vengono ricavati durata per epoca, posizione del best epoch, rebound della validation loss,
generalization gap e statistiche dei gradienti.

In [ ]:
def nested(mapping, *keys, default=np.nan):
    current = mapping
    for key in keys:
        if not isinstance(current, dict) or key not in current:
            return default
        current = current[key]
    return current


def finite_number(value):
    try:
        number = float(value)
    except (TypeError, ValueError):
        return np.nan
    return number if np.isfinite(number) else np.nan


def configuration_label(model, qubits):
    if model in {"B0", "B1"} or pd.isna(qubits):
        return str(model)
    return f"{model}-{int(qubits)}"


histories = {}
summary_rows = []
integrity_rows = list(parse_issues)
required_artifacts = TARGET_FILES

for item in run_bundles:
    archive_label = item["archive"]
    run_path = item["run_path"]
    bundle = item["bundle"]
    run_uid = f"{archive_label}::{run_path}"
    missing = sorted(required_artifacts - set(bundle))
    for filename in missing:
        integrity_rows.append({
            "archive": archive_label, "run_path": run_path,
            "file": filename, "issue": "artefatto mancante",
        })

    config = bundle.get("run_config.json") or {}
    validation = bundle.get("metrics.json") or {}
    test = bundle.get("test_metrics.json") or {}
    history_rows = bundle.get("history.jsonl") or []
    history = pd.DataFrame(history_rows)
    histories[run_uid] = history

    model = str(config.get("model", "unknown"))
    qubits = finite_number(config.get("n_qubits"))
    task = str(config.get("task", nested(config, "dataset_protocol", "task", "name", default="unknown")))
    subset = normalize_subset(config.get("subset_size"))
    repeat_seed = config.get("repeat_seed")
    fold = config.get("fold")

    if history.empty:
        epochs_recorded = 0
        first_epoch = last_epoch_history = np.nan
        epoch_time_median = epoch_time_max = epoch_time_max_ratio = np.nan
        train_f1_final = validation_f1_final = np.nan
        train_validation_gap_final = validation_loss_rebound = np.nan
        gradient_median = gradient_max = gradient_max_ratio = np.nan
        quantum_gradient_median = quantum_gradient_min = quantum_gradient_max = np.nan
        quantum_gradient_nonzero_fraction = np.nan
        history_best_epoch = np.nan
        learning_rate_changes = np.nan
    else:
        history = history.sort_values("epoch", kind="stable")
        epochs_recorded = len(history)
        first_epoch = finite_number(history["epoch"].iloc[0])
        last_epoch_history = finite_number(history["epoch"].iloc[-1])
        epoch_times = pd.to_numeric(history.get("elapsed_seconds"), errors="coerce").dropna()
        epoch_time_median = float(epoch_times.median()) if len(epoch_times) else np.nan
        epoch_time_max = float(epoch_times.max()) if len(epoch_times) else np.nan
        epoch_time_max_ratio = (
            epoch_time_max / epoch_time_median
            if np.isfinite(epoch_time_median) and epoch_time_median > 0 else np.nan
        )
        train_f1 = pd.to_numeric(history.get("train_f1_macro"), errors="coerce")
        val_f1 = pd.to_numeric(history.get("validation_f1_macro"), errors="coerce")
        train_f1_final = finite_number(train_f1.iloc[-1]) if len(train_f1) else np.nan
        validation_f1_final = finite_number(val_f1.iloc[-1]) if len(val_f1) else np.nan
        train_validation_gap_final = train_f1_final - validation_f1_final
        val_loss = pd.to_numeric(history.get("validation_loss"), errors="coerce").dropna()
        validation_loss_rebound = (
            float(val_loss.iloc[-1] - val_loss.min()) if len(val_loss) else np.nan
        )
        gradients = pd.to_numeric(history.get("gradient_norm"), errors="coerce").dropna().abs()
        gradient_median = float(gradients.median()) if len(gradients) else np.nan
        gradient_max = float(gradients.max()) if len(gradients) else np.nan
        gradient_max_ratio = (
            gradient_max / gradient_median
            if np.isfinite(gradient_median) and gradient_median > 0 else np.nan
        )
        quantum_gradients = pd.to_numeric(
            history.get("quantum_gradient_norm"), errors="coerce"
        ).dropna().abs()
        quantum_gradient_median = float(quantum_gradients.median()) if len(quantum_gradients) else np.nan
        quantum_gradient_min = float(quantum_gradients.min()) if len(quantum_gradients) else np.nan
        quantum_gradient_max = float(quantum_gradients.max()) if len(quantum_gradients) else np.nan
        quantum_gradient_nonzero_fraction = (
            float((quantum_gradients > 1e-12).mean()) if len(quantum_gradients) else np.nan
        )
        best_mask = history.get("is_best", pd.Series(False, index=history.index)).fillna(False).astype(bool)
        history_best_epoch = (
            finite_number(history.loc[best_mask, "epoch"].iloc[-1])
            if best_mask.any() else np.nan
        )
        learning_rates = pd.to_numeric(history.get("learning_rate"), errors="coerce").dropna()
        learning_rate_changes = int((learning_rates.diff().fillna(0) != 0).sum())

    best_epoch = finite_number(nested(validation, "runtime", "best_epoch", default=history_best_epoch))
    last_epoch = finite_number(nested(validation, "runtime", "last_epoch", default=last_epoch_history))
    best_epoch_fraction = best_epoch / last_epoch if np.isfinite(last_epoch) and last_epoch > 0 else np.nan
    merger_f1 = finite_number(nested(test, "per_class", "merger", "f1"))
    elliptical_f1 = finite_number(nested(test, "per_class", "elliptical", "f1"))
    spiral_f1 = finite_number(nested(test, "per_class", "spiral", "f1"))
    other_class_mean = np.nanmean([elliptical_f1, spiral_f1])
    merger_gap = other_class_mean - merger_f1 if np.isfinite(merger_f1) else np.nan

    summary_rows.append({
        "run_uid": run_uid,
        "archive": archive_label,
        "run_path": run_path,
        "task": task,
        "model": model,
        "n_qubits": qubits,
        "configuration": configuration_label(model, qubits),
        "subset_size": subset,
        "repeat_seed": repeat_seed,
        "fold": fold,
        "iteration": config.get("iteration"),
        "config_fingerprint": config.get("config_fingerprint"),
        "requested_device": nested(config, "runtime", "requested_device", default=None),
        "effective_device": nested(config, "runtime", "effective_device", default=None),
        "python_version": nested(config, "environment", "python", default=None),
        "validation_f1_macro": finite_number(nested(validation, "f1", "macro")),
        "test_f1_macro": finite_number(nested(test, "f1", "macro")),
        "validation_test_gap": (
            finite_number(nested(test, "f1", "macro"))
            - finite_number(nested(validation, "f1", "macro"))
        ),
        "test_accuracy": finite_number(test.get("accuracy")),
        "test_balanced_accuracy": finite_number(test.get("balanced_accuracy")),
        "test_log_loss": finite_number(test.get("log_loss")),
        "test_brier_score": finite_number(test.get("brier_score")),
        "test_ece": finite_number(test.get("ece")),
        "elliptical_f1": elliptical_f1,
        "spiral_f1": spiral_f1,
        "merger_precision": finite_number(nested(test, "per_class", "merger", "precision")),
        "merger_recall": finite_number(nested(test, "per_class", "merger", "recall")),
        "merger_f1": merger_f1,
        "merger_gap_to_other_classes": merger_gap,
        "n_samples_test": finite_number(test.get("n_samples")),
        "epochs_recorded": epochs_recorded,
        "first_epoch": first_epoch,
        "last_epoch": last_epoch,
        "best_epoch": best_epoch,
        "best_epoch_fraction": best_epoch_fraction,
        "stopped_early": nested(validation, "runtime", "stopped_early", default=None),
        "learning_rate_changes": learning_rate_changes,
        "training_seconds": finite_number(nested(test, "runtime", "training_seconds", default=nested(validation, "runtime", "training_seconds"))),
        "inference_seconds": finite_number(nested(test, "runtime", "inference_seconds", default=nested(validation, "runtime", "inference_seconds"))),
        "epoch_time_median": epoch_time_median,
        "epoch_time_max": epoch_time_max,
        "epoch_time_max_ratio": epoch_time_max_ratio,
        "train_f1_final": train_f1_final,
        "history_validation_f1_final": validation_f1_final,
        "train_validation_gap_final": train_validation_gap_final,
        "validation_loss_rebound": validation_loss_rebound,
        "gradient_median": gradient_median,
        "gradient_max": gradient_max,
        "gradient_max_ratio": gradient_max_ratio,
        "quantum_gradient_median": quantum_gradient_median,
        "quantum_gradient_min": quantum_gradient_min,
        "quantum_gradient_max": quantum_gradient_max,
        "quantum_gradient_nonzero_fraction": quantum_gradient_nonzero_fraction,
    })

run_summary = pd.DataFrame(summary_rows)
run_summary["subset_size"] = pd.Categorical(
    run_summary["subset_size"], categories=REGIME_ORDER, ordered=True
)
if run_summary.duplicated(["task", "configuration", "subset_size", "repeat_seed", "fold"]).any():
    raise ValueError("Sono presenti run duplicate nella chiave sperimentale.")
print({
    "runs": len(run_summary),
    "histories": len(histories),
    "integrity_issues": len(integrity_rows),
    "by_subset": run_summary["subset_size"].astype(str).value_counts().to_dict(),
})
display(run_summary.head())

## 3. Regole di segnalazione

Per le metriche continue viene usato lo z-score robusto
`0,6745 × (x − mediana) / MAD`. La soglia `|z| ≥ 3,5` viene applicata all'interno della
stessa configurazione sperimentale. Se il MAD è nullo viene usata una scala basata sull'IQR;
se anche l'IQR è nullo il gruppo non genera falsi outlier.

Le regole assolute riguardano soltanto condizioni interpretabili senza confronto: valori non
finiti, metriche fuori `[0,1]`, epoche oltre cinque volte la mediana della stessa run e
gradienti quantistici nulli in un modello **Q addestrabile**.

In [ ]:
def robust_z(values):
    series = pd.to_numeric(values, errors="coerce").astype(float)
    median = series.median()
    mad = (series - median).abs().median()
    if np.isfinite(mad) and mad > 0:
        return 0.67448975 * (series - median) / mad
    q1, q3 = series.quantile([0.25, 0.75])
    robust_sigma = (q3 - q1) / 1.349
    if np.isfinite(robust_sigma) and robust_sigma > 0:
        return (series - median) / robust_sigma
    return pd.Series(0.0, index=series.index)


homogeneous_keys = ["task", "subset_size", "configuration"]
for metric in (
    "test_f1_macro", "validation_test_gap", "test_ece", "training_seconds",
    "inference_seconds", "train_validation_gap_final", "validation_loss_rebound",
):
    run_summary[f"{metric}_robust_z"] = run_summary.groupby(
        homogeneous_keys, observed=True, dropna=False
    )[metric].transform(robust_z)

flags_list = []


def add_flag(row, category, severity, metric, value, reference, reason):
    flags_list.append({
        "run_uid": row["run_uid"],
        "archive": row["archive"],
        "run_path": row["run_path"],
        "task": row["task"],
        "model": row["model"],
        "n_qubits": row["n_qubits"],
        "configuration": row["configuration"],
        "subset_size": str(row["subset_size"]),
        "repeat_seed": row["repeat_seed"],
        "category": category,
        "severity": severity,
        "metric": metric,
        "value": value,
        "reference": reference,
        "reason": reason,
    })


for issue in integrity_rows:
    matching = run_summary.loc[
        run_summary["archive"].eq(issue.get("archive"))
        & run_summary["run_path"].eq(issue.get("run_path"))
    ]
    if len(matching):
        row = matching.iloc[0]
        add_flag(
            row, "integrity", "critical", issue.get("file", "JSON"), np.nan,
            "artefatto leggibile e presente", issue.get("issue", "problema di integrità"),
        )

bounded_metrics = (
    "validation_f1_macro", "test_f1_macro", "test_accuracy",
    "test_balanced_accuracy", "test_brier_score", "test_ece",
    "elliptical_f1", "spiral_f1", "merger_precision", "merger_recall", "merger_f1",
)
for _, row in run_summary.iterrows():
    for metric in ("validation_f1_macro", "test_f1_macro", "training_seconds", "epochs_recorded"):
        if not np.isfinite(finite_number(row[metric])):
            add_flag(row, "integrity", "critical", metric, row[metric], "valore finito", "valore mancante o non finito")
    for metric in bounded_metrics:
        value = finite_number(row[metric])
        if np.isfinite(value) and not 0 <= value <= 1:
            add_flag(row, "integrity", "critical", metric, value, "intervallo [0, 1]", "metrica fuori intervallo")

    performance_z = finite_number(row["test_f1_macro_robust_z"])
    if abs(performance_z) >= 3.5:
        direction = "inferiore" if performance_z < 0 else "superiore"
        add_flag(
            row, "performance", "high", "test_f1_macro", row["test_f1_macro"],
            f"z robusto={performance_z:.2f}",
            f"Macro-F1 {direction} alle run omogenee della stessa cella sperimentale",
        )

    gap_z = finite_number(row["validation_test_gap_robust_z"])
    if abs(gap_z) >= 3.5 or abs(finite_number(row["validation_test_gap"])) >= 0.05:
        add_flag(
            row, "validation_test", "high", "validation_test_gap",
            row["validation_test_gap"], f"z robusto={gap_z:.2f}",
            "discrepanza insolita tra Macro-F1 di validation e test",
        )

    ece_z = finite_number(row["test_ece_robust_z"])
    if ece_z >= 3.5:
        add_flag(row, "calibration", "medium", "test_ece", row["test_ece"], f"z robusto={ece_z:.2f}", "ECE insolitamente elevato nella cella sperimentale")

    if finite_number(row["epoch_time_max_ratio"]) >= 5:
        add_flag(
            row, "epoch_time", "high", "epoch_time_max_ratio",
            row["epoch_time_max_ratio"], "soglia 5×",
            "almeno un'epoca è molto più lenta della mediana della stessa run",
        )
    training_z = finite_number(row["training_seconds_robust_z"])
    if training_z >= 3.5:
        add_flag(row, "cost", "medium", "training_seconds", row["training_seconds"], f"z robusto={training_z:.2f}", "training insolitamente lento rispetto alla stessa configurazione")
    inference_z = finite_number(row["inference_seconds_robust_z"])
    if inference_z >= 3.5:
        add_flag(row, "cost", "medium", "inference_seconds", row["inference_seconds"], f"z robusto={inference_z:.2f}", "inferenza insolitamente lenta rispetto alla stessa configurazione")

    if finite_number(row["train_validation_gap_final"]) >= 0.10:
        add_flag(row, "training_dynamics", "medium", "train_validation_gap_final", row["train_validation_gap_final"], "soglia 0,10", "generalization gap finale elevato")
    if finite_number(row["validation_loss_rebound"]) >= 0.20:
        add_flag(row, "training_dynamics", "medium", "validation_loss_rebound", row["validation_loss_rebound"], "soglia 0,20", "validation loss finale molto sopra il minimo raggiunto")
    if finite_number(row["gradient_max_ratio"]) >= 10:
        add_flag(row, "gradient", "medium", "gradient_max_ratio", row["gradient_max_ratio"], "soglia 10×", "picco del gradiente rispetto alla mediana della run")

    # Gli zeri strutturali di B0/B1/B2 e del circuito congelato QF non sono anomalie.
    if row["model"] == "Q" and finite_number(row["quantum_gradient_nonzero_fraction"]) == 0:
        add_flag(
            row, "quantum_gradient", "critical", "quantum_gradient_nonzero_fraction",
            row["quantum_gradient_nonzero_fraction"], "atteso > 0 per Q addestrabile",
            "gradienti dei parametri quantistici nulli in tutte le epoche registrate",
        )

    if np.isfinite(finite_number(row["merger_gap_to_other_classes"])) and row["merger_gap_to_other_classes"] >= 0.15:
        add_flag(
            row, "merger", "medium", "merger_gap_to_other_classes",
            row["merger_gap_to_other_classes"], "soglia 0,15",
            "F1 merger molto inferiore alla media di elliptical e spiral",
        )

flags = pd.DataFrame(flags_list)
if flags.empty:
    flags = pd.DataFrame(columns=[
        "run_uid", "archive", "run_path", "task", "model", "n_qubits",
        "configuration", "subset_size", "repeat_seed", "category", "severity",
        "metric", "value", "reference", "reason",
    ])
flags["severity_score"] = flags["severity"].map(SEVERITY_SCORE).fillna(0).astype(int)
display(flags["category"].value_counts().rename_axis("categoria").to_frame("segnalazioni"))

## 4. Confronti appaiati e influenza dei seed

Oltre agli outlier assoluti, vengono costruiti i confronti Q−B2, Q−QF, 8−4 qubit e
tre-classi−binario. Per ogni seed si calcola anche quanto cambia la media del gruppo se quel
seed viene rimosso. Queste quantità servono a trovare casi narrativamente interessanti, non a
ridefinire le ipotesi statistiche della tesi.

In [ ]:
def matched_difference(frame, *, contrast, dimension, value_a, value_b, keys, label_a, label_b):
    a = frame.loc[dimension(frame, value_a)].copy()
    b = frame.loc[dimension(frame, value_b)].copy()
    keep = [*keys, "run_uid", "run_path", "test_f1_macro"]
    a = a[keep].rename(columns={
        "run_uid": "run_uid_a", "run_path": "run_path_a", "test_f1_macro": "value_a"
    })
    b = b[keep].rename(columns={
        "run_uid": "run_uid_b", "run_path": "run_path_b", "test_f1_macro": "value_b"
    })
    paired = a.merge(b, on=keys, how="inner", validate="one_to_one")
    paired["contrast"] = contrast
    paired["label_a"] = label_a
    paired["label_b"] = label_b
    paired["effect_pp"] = 100 * (paired["value_a"] - paired["value_b"])
    return paired


contrasts = []
base_keys = ["task", "subset_size", "n_qubits", "repeat_seed", "fold"]
contrasts.append(matched_difference(
    run_summary, contrast="Q-B2", dimension=lambda frame, value: frame["model"].eq(value),
    value_a="Q", value_b="B2", keys=base_keys, label_a="Q", label_b="B2",
))
contrasts.append(matched_difference(
    run_summary, contrast="Q-QF", dimension=lambda frame, value: frame["model"].eq(value),
    value_a="Q", value_b="QF", keys=base_keys, label_a="Q", label_b="QF",
))

for model in ("B2", "QF", "Q"):
    model_frame = run_summary.loc[run_summary["model"].eq(model)]
    contrasts.append(matched_difference(
        model_frame,
        contrast=f"8q-4q:{model}",
        dimension=lambda frame, value: frame["n_qubits"].eq(value),
        value_a=8, value_b=4,
        keys=["task", "subset_size", "model", "repeat_seed", "fold"],
        label_a=f"{model}-8", label_b=f"{model}-4",
    ))

for model in ("B2", "QF", "Q"):
    for qubits in (4, 8):
        model_frame = run_summary.loc[
            run_summary["model"].eq(model) & run_summary["n_qubits"].eq(qubits)
        ]
        contrasts.append(matched_difference(
            model_frame,
            contrast=f"three-binary:{model}-{qubits}",
            dimension=lambda frame, value: frame["task"].eq(value),
            value_a="three_class", value_b="binary",
            keys=["subset_size", "model", "n_qubits", "repeat_seed", "fold"],
            label_a="three_class", label_b="binary",
        ))

matched_contrasts = pd.concat(contrasts, ignore_index=True, sort=False)
contrast_group_keys = [
    column for column in ("contrast", "task", "subset_size", "n_qubits", "model")
    if column in matched_contrasts.columns
]
matched_contrasts["effect_robust_z"] = matched_contrasts.groupby(
    contrast_group_keys, dropna=False, observed=True
)["effect_pp"].transform(robust_z)
group_mean = matched_contrasts.groupby(
    contrast_group_keys, dropna=False, observed=True
)["effect_pp"].transform("mean")
group_count = matched_contrasts.groupby(
    contrast_group_keys, dropna=False, observed=True
)["effect_pp"].transform("count")
leave_one_out = (
    (group_mean * group_count - matched_contrasts["effect_pp"])
    / (group_count - 1).replace(0, np.nan)
)
matched_contrasts["group_mean_pp"] = group_mean
matched_contrasts["leave_one_out_mean_pp"] = leave_one_out
matched_contrasts["seed_influence_pp"] = group_mean - leave_one_out

contrast_flags = matched_contrasts.loc[
    matched_contrasts["effect_robust_z"].abs().ge(3.5)
    | matched_contrasts["seed_influence_pp"].abs().ge(0.5)
]
for _, case in contrast_flags.iterrows():
    matching = run_summary.loc[run_summary["run_uid"].eq(case["run_uid_a"])]
    if len(matching):
        row = matching.iloc[0]
        add_flag(
            row, "paired_effect", "medium", case["contrast"], case["effect_pp"],
            f"influenza seed={case['seed_influence_pp']:.2f} pp",
            "effetto appaiato o influenza del seed insolitamente elevati",
        )

# Ricostruisce flags dopo l'aggiunta dei casi appaiati.
flags = pd.DataFrame(flags_list)
flags["severity_score"] = flags["severity"].map(SEVERITY_SCORE).fillna(0).astype(int)

## 5. Andamenti non monotoni rispetto alle risorse

Una riduzione superiore a un punto percentuale passando a un training set più grande viene
segnalata come caso da verificare. Non è automaticamente un errore: con soli dieci seed può
rappresentare variabilità di ottimizzazione, sensibilità al subset o un caso genuinamente
informativo.

In [ ]:
non_monotonic_rows = []
resource_keys = ["task", "configuration", "repeat_seed", "fold"]
for keys, group in run_summary.groupby(resource_keys, observed=True, dropna=False):
    lookup = {str(row.subset_size): row for row in group.itertuples(index=False)}
    for smaller, larger in zip(REGIME_ORDER[:-1], REGIME_ORDER[1:]):
        if smaller not in lookup or larger not in lookup:
            continue
        small_row, large_row = lookup[smaller], lookup[larger]
        change_pp = 100 * (large_row.test_f1_macro - small_row.test_f1_macro)
        if change_pp < -1.0:
            record = dict(zip(resource_keys, keys if isinstance(keys, tuple) else (keys,)))
            record.update({
                "smaller_regime": smaller,
                "larger_regime": larger,
                "f1_smaller": small_row.test_f1_macro,
                "f1_larger": large_row.test_f1_macro,
                "change_pp": change_pp,
                "run_uid_larger": large_row.run_uid,
                "run_path_larger": large_row.run_path,
            })
            non_monotonic_rows.append(record)
            matching = run_summary.loc[run_summary["run_uid"].eq(large_row.run_uid)]
            if len(matching):
                add_flag(
                    matching.iloc[0], "resource_monotonicity", "medium",
                    f"{smaller}->{larger}", change_pp, "atteso non inferiore a -1 pp",
                    "Macro-F1 diminuisce aumentando il numero di immagini di training",
                )

non_monotonic = pd.DataFrame(non_monotonic_rows)
if non_monotonic.empty:
    non_monotonic = pd.DataFrame(columns=[
        *resource_keys, "smaller_regime", "larger_regime", "f1_smaller",
        "f1_larger", "change_pp", "run_uid_larger", "run_path_larger",
    ])

flags = pd.DataFrame(flags_list)
flags["severity_score"] = flags["severity"].map(SEVERITY_SCORE).fillna(0).astype(int)

if len(flags):
    ranking = flags.groupby("run_uid", as_index=False).agg(
        anomaly_score=("severity_score", "sum"),
        flag_count=("category", "size"),
        categories=("category", lambda values: ", ".join(sorted(set(values)))),
        reasons=("reason", lambda values: " | ".join(dict.fromkeys(values))),
    )
    ranked_runs = run_summary.merge(ranking, on="run_uid", how="left")
else:
    ranked_runs = run_summary.copy()
    ranked_runs["anomaly_score"] = 0
    ranked_runs["flag_count"] = 0
    ranked_runs["categories"] = ""
    ranked_runs["reasons"] = ""
ranked_runs[["anomaly_score", "flag_count"]] = ranked_runs[
    ["anomaly_score", "flag_count"]
].fillna(0)
ranked_runs = ranked_runs.sort_values(
    ["anomaly_score", "flag_count", "test_f1_macro"],
    ascending=[False, False, True], kind="stable",
)

def save_table(frame, name):
    path = TABLE_DIR / f"{name}.csv"
    frame.to_csv(path, index=False, float_format="%.8g")
    EXPORTED_TABLES.append(path)
    return path


save_table(run_summary, "run_summary")
save_table(flags, "flags")
save_table(ranked_runs, "ranked_runs")
save_table(matched_contrasts, "matched_contrasts")
save_table(matched_contrasts.sort_values("seed_influence_pp", key=lambda x: x.abs(), ascending=False), "seed_influence")
save_table(non_monotonic, "non_monotonic_resources")

display(ranked_runs[[
    "task", "configuration", "subset_size", "repeat_seed", "test_f1_macro",
    "anomaly_score", "categories", "run_path",
]].head(15))

# 6. Grafici diagnostici

Le figure successive servono per scegliere casi da discutere. Mantengono punti individuali,
scale comuni nei pannelli confrontabili e colori coerenti con il notebook delle figure finali.

In [ ]:
def clean_axis(axis, grid_axis="y"):
    axis.grid(True, axis=grid_axis, color="#D9D9D9", linewidth=0.7, alpha=0.65)
    axis.grid(False, axis="x" if grid_axis == "y" else "y")


def panel_label(axis, label):
    axis.text(-0.10, 1.04, label, transform=axis.transAxes,
              fontsize=12, fontweight="bold", va="bottom")


def save_figure(figure, name):
    png = FIGURE_DIR / f"{name}.png"
    pdf = FIGURE_DIR / f"{name}.pdf"
    figure.savefig(png, dpi=EXPORT_DPI, bbox_inches="tight", facecolor="white")
    figure.savefig(pdf, bbox_inches="tight", facecolor="white")
    display(figure)
    plt.close(figure)
    EXPORTED_FIGURES.append({"name": name, "png": str(png), "pdf": str(pdf)})
    print(f"Salvati: {png.name}, {pdf.name}")


# 01 — conteggio delle segnalazioni, senza sommare categorie incompatibili.
if len(flags):
    overview = flags.pivot_table(
        index="category", columns="task", values="run_uid", aggfunc="count", fill_value=0
    )
else:
    overview = pd.DataFrame(0, index=["nessuna"], columns=["binary", "three_class"])
figure, axis = plt.subplots(figsize=(8.2, max(3.8, 0.48 * len(overview) + 1.5)))
sns.heatmap(overview, annot=True, fmt="g", cmap="Blues", cbar=False,
            linewidths=0.8, linecolor="white", ax=axis)
axis.set_xlabel("Task")
axis.set_ylabel("Categoria della segnalazione")
axis.set_xticklabels([TASK_LABELS.get(str(x.get_text()), x.get_text()) for x in axis.get_xticklabels()])
axis.set_title("Dove si concentrano le run da approfondire?")
figure.tight_layout()
save_figure(figure, "01_flag_overview")

In [ ]:
# 02 — distribuzioni di Macro-F1 e outlier robusti evidenziati.
performance_flag_uids = set(flags.loc[flags["category"].eq("performance"), "run_uid"])
figure, axes = plt.subplots(1, 2, figsize=(13.2, 5.2), sharey=False)
for panel_index, task in enumerate(("binary", "three_class")):
    axis = axes[panel_index]
    panel = run_summary.loc[run_summary["task"].eq(task)].copy()
    for x_index, regime in enumerate(REGIME_ORDER):
        values = panel.loc[panel["subset_size"].astype(str).eq(regime)]
        x = x_index + np.linspace(-0.22, 0.22, len(values))
        colors = ["#D0D0D0" if uid not in performance_flag_uids else "#D55E00" for uid in values["run_uid"]]
        sizes = [12 if uid not in performance_flag_uids else 42 for uid in values["run_uid"]]
        axis.scatter(x, values["test_f1_macro"], c=colors, s=sizes, alpha=0.78,
                     edgecolors="white", linewidths=0.35)
    flagged_panel = panel.loc[panel["run_uid"].isin(performance_flag_uids)]
    for row in flagged_panel.itertuples(index=False):
        x = REGIME_ORDER.index(str(row.subset_size))
        axis.annotate(
            f"{row.configuration}/s{row.repeat_seed}",
            (x, row.test_f1_macro), xytext=(4, 5), textcoords="offset points",
            fontsize=7, color="#A33A00",
        )
    axis.set_xticks(range(4), [REGIME_LABELS[x] for x in REGIME_ORDER])
    axis.set_xlabel("Immagini di training")
    axis.set_ylabel("Macro-F1 sul test")
    axis.set_title(TASK_LABELS[task])
    clean_axis(axis)
    panel_label(axis, chr(ord("a") + panel_index))
figure.suptitle("Outlier prestazionali all'interno delle celle sperimentali",
                fontsize=14, fontweight="semibold", y=1.02)
figure.text(0.5, -0.01, "Grigio: run non segnalate · arancione: |z robusto| ≥ 3,5",
            ha="center", fontsize=9, color="#444444")
figure.tight_layout()
save_figure(figure, "02_performance_outliers")

In [ ]:
# 03 — dinamica: posizione del best epoch, generalization gap e rebound della loss.
figure, axes = plt.subplots(1, 2, figsize=(13.2, 5.2))
for model, group in run_summary.groupby("model", sort=False):
    color = MODEL_COLORS.get(model, "#777777")
    axes[0].scatter(group["best_epoch_fraction"], group["train_validation_gap_final"],
                    s=18, color=color, alpha=0.45, label=model, edgecolors="none")
    axes[1].scatter(group["validation_loss_rebound"], group["test_f1_macro_robust_z"],
                    s=18, color=color, alpha=0.45, label=model, edgecolors="none")
axes[0].axhline(0, color="#222222", linewidth=1)
axes[0].set(xlabel="Best epoch / ultima epoca", ylabel="Train − validation Macro-F1 finale",
            title="Quando arriva il best epoch e quanto è ampio il gap?")
axes[1].axhline(0, color="#222222", linewidth=1)
axes[1].axhline(-3.5, color="#D55E00", linewidth=1, linestyle="--")
axes[1].set(xlabel="Validation loss finale − minimo", ylabel="z robusto della Macro-F1 test",
            title="Rebound della loss e prestazione anomala")
for index, axis in enumerate(axes):
    clean_axis(axis)
    panel_label(axis, chr(ord("a") + index))
axes[1].legend(loc="best", ncol=3)
figure.suptitle("Dinamiche di training da controllare", fontsize=14,
                fontweight="semibold", y=1.02)
figure.tight_layout()
save_figure(figure, "03_training_dynamics")

In [ ]:
# 04 — confronto Q−B2 per seed: distribuzione e influenza leave-one-out.
q_b2 = matched_contrasts.loc[matched_contrasts["contrast"].eq("Q-B2")].copy()
figure, axes = plt.subplots(1, 2, figsize=(13.2, 5.2))
positions = {regime: index for index, regime in enumerate(REGIME_ORDER)}
for task, marker in (("binary", "o"), ("three_class", "s")):
    task_data = q_b2.loc[q_b2["task"].eq(task)]
    for qubits, color, offset in ((4, "#0072B2", -0.08), (8, "#7A5195", 0.08)):
        data = task_data.loc[task_data["n_qubits"].eq(qubits)]
        xs = np.array([positions[str(x)] + offset for x in data["subset_size"]])
        axes[0].scatter(xs, data["effect_pp"], s=20, color=color, marker=marker,
                        alpha=0.40, edgecolors="none")
        axes[1].scatter(xs, data["seed_influence_pp"].abs(), s=20, color=color,
                        marker=marker, alpha=0.40, edgecolors="none")
axes[0].axhline(0, color="#222222", linewidth=1)
axes[0].set(ylabel="Q − B2 (punti percentuali)", title="Effetti appaiati dei singoli seed")
axes[1].set(ylabel="|Influenza leave-one-out| (punti percentuali)",
            title="Quanto cambia la media rimuovendo un seed?")
for index, axis in enumerate(axes):
    axis.set_xticks(range(4), [REGIME_LABELS[x] for x in REGIME_ORDER])
    axis.set_xlabel("Immagini di training")
    clean_axis(axis)
    panel_label(axis, chr(ord("a") + index))
legend = [
    Line2D([0], [0], color="#666666", marker="o", linestyle="", label="Binario"),
    Line2D([0], [0], color="#666666", marker="s", linestyle="", label="Tre classi"),
    Line2D([0], [0], color="#0072B2", marker="o", label="4 qubit"),
    Line2D([0], [0], color="#7A5195", marker="o", label="8 qubit"),
]
axes[1].legend(handles=legend, ncol=2)
figure.suptitle("Seed estremi e sensibili nel confronto centrale Q−B2",
                fontsize=14, fontweight="semibold", y=1.02)
figure.tight_layout()
save_figure(figure, "04_matched_contrasts")

In [ ]:
# 05 — traiettorie delle configurazioni che peggiorano aumentando i dati.
selected_keys = non_monotonic.sort_values("change_pp").head(10)[
    ["task", "configuration", "repeat_seed", "fold"]
].drop_duplicates()
if selected_keys.empty:
    piv = run_summary.pivot(
        index=["task", "configuration", "repeat_seed", "fold"],
        columns="subset_size", values="test_f1_macro"
    ).dropna()
    piv["worst_step"] = piv[list(REGIME_ORDER)].diff(axis=1).min(axis=1)
    selected_keys = piv.nsmallest(6, "worst_step").reset_index()[
        ["task", "configuration", "repeat_seed", "fold"]
    ]
figure, axes = plt.subplots(1, 2, figsize=(13.2, 5.2), sharey=False)
for panel_index, task in enumerate(("binary", "three_class")):
    axis = axes[panel_index]
    task_keys = selected_keys.loc[selected_keys["task"].eq(task)].head(6)
    for case in task_keys.itertuples(index=False):
        trajectory = run_summary.loc[
            run_summary["task"].eq(task)
            & run_summary["configuration"].eq(case.configuration)
            & run_summary["repeat_seed"].eq(case.repeat_seed)
            & run_summary["fold"].eq(case.fold)
        ].sort_values("subset_size")
        axis.plot(
            range(len(trajectory)), trajectory["test_f1_macro"], marker="o", linewidth=1.7,
            label=f"{case.configuration}/s{case.repeat_seed}",
            color=MODEL_COLORS.get(str(case.configuration).split("-")[0], "#777777"),
            alpha=0.85,
        )
    axis.set_xticks(range(4), [REGIME_LABELS[x] for x in REGIME_ORDER])
    axis.set_xlabel("Immagini di training")
    axis.set_ylabel("Macro-F1 sul test")
    axis.set_title(TASK_LABELS[task])
    clean_axis(axis)
    panel_label(axis, chr(ord("a") + panel_index))
    if len(task_keys):
        axis.legend(fontsize=7, loc="best")
figure.suptitle("Casi non monotoni rispetto alla numerosità del training",
                fontsize=14, fontweight="semibold", y=1.02)
figure.tight_layout()
save_figure(figure, "05_resource_trajectories")

In [ ]:
# 06 — storie delle run più segnalate, utili per l'ispezione qualitativa.
selected_cases = ranked_runs.loc[ranked_runs["anomaly_score"].gt(0)].head(TOP_CASES)
if selected_cases.empty:
    selected_cases = ranked_runs.head(TOP_CASES)
rows = math.ceil(max(1, len(selected_cases)) / 2)
figure, axes = plt.subplots(rows, 2, figsize=(13.2, 3.4 * rows), squeeze=False)
flat_axes = axes.ravel()
for axis, case in zip(flat_axes, selected_cases.itertuples(index=False)):
    history = histories.get(case.run_uid, pd.DataFrame()).sort_values("epoch")
    if history.empty:
        axis.text(0.5, 0.5, "History mancante", ha="center", va="center")
    else:
        axis.plot(history["epoch"], history["train_f1_macro"], color="#666666",
                  linestyle="--", label="Train F1")
        axis.plot(history["epoch"], history["validation_f1_macro"], color="#0072B2",
                  label="Validation F1")
        if np.isfinite(case.best_epoch):
            axis.axvline(case.best_epoch, color="#D55E00", linewidth=1,
                         linestyle=":", label="Best epoch")
    axis.set_title(
        f"{case.task} · {case.configuration} · {case.subset_size} · seed {case.repeat_seed}\n"
        f"score={int(case.anomaly_score)} · {str(case.categories)[:70]}",
        fontsize=9,
    )
    axis.set_xlabel("Epoca")
    axis.set_ylabel("Macro-F1")
    clean_axis(axis)
    axis.legend(fontsize=7, ncol=3, loc="lower right")
for axis in flat_axes[len(selected_cases):]:
    axis.set_visible(False)
figure.suptitle("History delle run con priorità di ispezione più alta",
                fontsize=14, fontweight="semibold", y=1.01)
figure.tight_layout()
save_figure(figure, "06_case_histories")

In [ ]:
# 07 — costi individuali; asse log per non nascondere code lunghe.
cost_uids = set(flags.loc[flags["category"].isin(("cost", "epoch_time")), "run_uid"])
figure, axes = plt.subplots(1, 2, figsize=(13.2, 5.2))
for panel_index, metric in enumerate(("training_seconds", "inference_seconds")):
    axis = axes[panel_index]
    for x_index, regime in enumerate(REGIME_ORDER):
        panel = run_summary.loc[run_summary["subset_size"].astype(str).eq(regime)]
        x = x_index + np.linspace(-0.22, 0.22, len(panel))
        colors = ["#BDBDBD" if uid not in cost_uids else "#D55E00" for uid in panel["run_uid"]]
        sizes = [12 if uid not in cost_uids else 38 for uid in panel["run_uid"]]
        axis.scatter(x, panel[metric], c=colors, s=sizes, alpha=0.72,
                     edgecolors="white", linewidths=0.3)
    axis.set_yscale("log")
    axis.set_xticks(range(4), [REGIME_LABELS[x] for x in REGIME_ORDER])
    axis.set_xlabel("Immagini di training")
    axis.set_ylabel("Secondi (scala log)")
    axis.set_title("Training totale" if metric == "training_seconds" else "Inferenza")
    clean_axis(axis)
    panel_label(axis, chr(ord("a") + panel_index))
figure.suptitle("Run con costo computazionale insolito",
                fontsize=14, fontweight="semibold", y=1.02)
figure.text(0.5, -0.01, "Arancione: segnalazione di costo o singola epoca ≥5× la mediana della run",
            ha="center", fontsize=9, color="#444444")
figure.tight_layout()
save_figure(figure, "07_cost_outliers")

# 7. Rapporto automatico e file di consegna

Il rapporto distingue esplicitamente errori tecnici, anomalie statistiche e casi scientifici.
I percorsi indicati sono percorsi interni ai TAR e permettono di recuperare esattamente la run.

In [ ]:
def markdown_table(frame, columns, limit=10):
    selected = frame.loc[:, columns].head(limit).copy()
    if selected.empty:
        return "_Nessun caso rilevato._"
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join("---" for _ in columns) + " |"
    rows = []
    for record in selected.to_dict("records"):
        values = []
        for column in columns:
            value = record[column]
            if isinstance(value, float):
                value = f"{value:.4g}"
            values.append(str(value).replace("|", "/"))
        rows.append("| " + " | ".join(values) + " |")
    return "\n".join([header, separator, *rows])


top_flags = ranked_runs.loc[ranked_runs["anomaly_score"].gt(0)]
most_influential = matched_contrasts.sort_values(
    "seed_influence_pp", key=lambda values: values.abs(), ascending=False
)
report_parts = [
    "# Casi particolari individuati nelle run",
    "",
    "> **Avvertenza metodologica.** Le segnalazioni sono strumenti esplorativi. "
    "Non costituiscono criteri automatici di esclusione e non sostituiscono i confronti "
    "statistici predefiniti della tesi.",
    "",
    "## Copertura dell'audit",
    "",
    f"- Run lette: **{len(run_summary)}**",
    f"- History lette: **{len(histories)}**",
    f"- Segnalazioni complessive: **{len(flags)}**",
    f"- Run con almeno una segnalazione: **{flags['run_uid'].nunique() if len(flags) else 0}**",
    f"- Transizioni non monotone oltre 1 pp: **{len(non_monotonic)}**",
    "",
    "## Run con priorità di ispezione",
    "",
    markdown_table(
        top_flags,
        ["task", "configuration", "subset_size", "repeat_seed", "test_f1_macro", "anomaly_score", "categories", "run_path"],
        limit=15,
    ),
    "",
    "## Seed più influenti nei confronti appaiati",
    "",
    markdown_table(
        most_influential,
        ["contrast", "subset_size", "repeat_seed", "effect_pp", "seed_influence_pp", "run_path_a"],
        limit=15,
    ),
    "",
    "## Peggioramenti aumentando i dati",
    "",
    markdown_table(
        non_monotonic.sort_values("change_pp"),
        ["task", "configuration", "repeat_seed", "smaller_regime", "larger_regime", "change_pp", "run_path_larger"],
        limit=15,
    ),
    "",
    "## Come usare questi casi nella tesi",
    "",
    "1. Scegliere un caso rappresentativo vicino alla mediana e uno o due casi estremi.",
    "2. Verificare la history prima di attribuire l'effetto all'architettura.",
    "3. Distinguere instabilità di ottimizzazione, difficoltà della classe merger e costo del simulatore.",
    "4. Presentare i casi per illustrare la variabilità, non come prova statistica autonoma.",
    "5. Non rimuovere una run salvo evidenza documentata di errore tecnico o artefatto corrotto.",
]
report_path = OUTPUT_DIR / "interesting_cases.md"
report_path.write_text("\n".join(report_parts), encoding="utf-8")

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "inputs": {
        "full_archive": str(FULL_ARCHIVE),
        "subset_archive": str(SUBSET_ARCHIVE),
    },
    "audit": {
        "runs": int(len(run_summary)),
        "histories": int(len(histories)),
        "flags": int(len(flags)),
        "flagged_runs": int(flags["run_uid"].nunique()) if len(flags) else 0,
        "non_monotonic_transitions": int(len(non_monotonic)),
        "robust_z_threshold": 3.5,
        "resource_drop_threshold_pp": -1.0,
        "automatic_exclusion": False,
    },
    "versions": {
        "python": sys.version.split()[0],
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "matplotlib": matplotlib.__version__,
        "seaborn": sns.__version__,
    },
    "figures": EXPORTED_FIGURES,
    "tables": [str(path) for path in EXPORTED_TABLES],
    "report": str(report_path),
}
manifest_path = OUTPUT_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

zip_path = None
if MAKE_ZIP:
    zip_path = Path(shutil.make_archive(
        str(OUTPUT_DIR.parent / f"{OUTPUT_DIR.name}_bundle"), "zip", root_dir=OUTPUT_DIR
    ))

missing_outputs = [
    path for item in EXPORTED_FIGURES for path in (Path(item["png"]), Path(item["pdf"]))
    if not path.is_file() or path.stat().st_size == 0
]
if missing_outputs:
    raise RuntimeError(f"Figure mancanti o vuote: {missing_outputs}")

display(Markdown("\n".join([
    "## Audit completato",
    "",
    f"- Run analizzate: **{len(run_summary)}**",
    f"- Run segnalate: **{flags['run_uid'].nunique() if len(flags) else 0}**",
    f"- Figure: **{len(EXPORTED_FIGURES)}** coppie PNG/PDF",
    f"- Tabelle: **{len(EXPORTED_TABLES)}**",
    f"- Rapporto: `{report_path}`",
    f"- ZIP: `{zip_path}`" if zip_path else "- ZIP: disattivato",
])))
print({
    "output": str(OUTPUT_DIR),
    "runs": len(run_summary),
    "flags": len(flags),
    "figure_pairs": len(EXPORTED_FIGURES),
    "validation": "ok",
})